# Лабораторная работа 2. Эмпирический риск и метод наименьших квадратов

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 1 |
| Опора на лекции | лекция 1: функция потерь (опр. 1.9), эмпирический и истинный риск (опр. 1.10–1.11), утв. 1.12 о несмещённости, принцип ERM (опр. 1.13), нормальные уравнения МНК (теорема 1.20), примеры 1.5 и 1.21 |
| Трудоёмкость | 2 ч аудиторно + домашняя работа (3–4 ч) |

## Цель работы

Увидеть, что утверждения лекции 1 — не формальность: эмпирический риск действительно несмещённо оценивает истинный, но **только** для алгоритма, выбранного независимо от выборки. Разобрать, как выбор функции потерь определяет, что именно мы восстанавливаем, и воспроизвести численный пример МНК из конспекта.

## Как устроено занятие

Ноутбук разбирается в аудитории: код запускается и обсуждается по ходу.
Большая часть ячеек уже написана — их нужно **прочитать и запустить**,
разобравшись, что происходит и почему.

Ячейки, помеченные `# ✍ ЗАДАНИЕ НА СЕМИНАРЕ`, заполняются самостоятельно
прямо на занятии; их немного, и каждая занимает несколько строк. Ячейки
**Вывод** — тоже ваши: короткий ответ своими словами на поставленный вопрос.

Дома выполняется отдельный ноутбук `lab02_homework.ipynb` — там
задания крупнее и делать их нужно самому.

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже) — и на семинаре, и в домашней работе.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from scipy import optimize
from sklearn.linear_model import LinearRegression
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=2)
describe_variant(variant)

---
# Часть 1. Функция потерь задаёт, что мы восстанавливаем

Определение 1.9: потеря $\mathcal L(a,x,y)\ge0$ измеряет ошибку на одном объекте;
в регрессии она зависит только от невязки $r = a(x) - y$. Эмпирический риск
(опр. 1.10) — её среднее по выборке:

$$
Q(a, X^\ell) = \frac1\ell\sum_{i=1}^{\ell}\mathcal L(a, x_i, y_i).
$$

Возьмём три потери и посмотрим на них как на функции невязки.

> **Напоминание — квантильная потеря и потеря Хьюбера.** **Квантильная** потеря штрафует перепрогноз и недопрогноз *разными* весами:
> $\mathcal L(r) = (1-q)\,r$ при $r\ge0$ и $-q\,r$ при $r<0$. При $q = 0.5$
> это половина абсолютной потери, при $q = 0.9$ недопрогноз в девять раз дороже
> перепрогноза. Нужна там, где ошибки несимметричны по цене: занизить запас
> товара на складе хуже, чем завысить.
>
> **Хьюбера** — компромисс между квадратичной и абсолютной: квадратичная при
> $|r|\le\delta$ и линейная дальше. Гладкая в нуле (в отличие от абсолютной,
> которую неудобно минимизировать градиентными методами) и не даёт выбросам
> раздувать функционал квадратично.

In [ ]:
def loss_squared(r):
    return r ** 2


def loss_absolute(r):
    return np.abs(r)


def loss_quantile(r, q=0.75):
    """Перепрогноз (r > 0) штрафуется весом 1 - q, недопрогноз -- весом q."""
    return np.where(r >= 0, (1.0 - q) * r, -q * r)

In [ ]:
r = np.linspace(-3, 3, 400)
fig, ax = plt.subplots()
for loss, name in [(loss_squared, "квадратичная"), (loss_absolute, "абсолютная"),
                   (loss_quantile, "квантильная, q=0.75")]:
    ax.plot(r, loss(r), lw=2, label=name)
ax.set_xlabel("невязка $r = a(x) - y$"); ax.set_ylabel(r"$\mathcal{L}(r)$")
ax.set_title("Три функции потерь"); ax.legend()
plt.tight_layout(); plt.show()

### Задание 1.1. Эмпирический риск

Простейшая модель алгоритмов — константы $a(x)\equiv c$. Принцип ERM (опр. 1.13)
требует найти $c^* = \arg\min_c Q(c)$. Начнём с самой функции $Q$.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def empirical_risk(loss, pred, y):
    """Q(a, X^l) -- среднее значение потерь по выборке."""
    # TODO (1 строка): среднее значение loss(pred - y)
    raise NotImplementedError


y_sample = rng.gamma(shape=2.0, scale=3.0, size=400)     # заведомо несимметричное
print(f"Q константы 5.0 при квадратичной потере: "
      f"{empirical_risk(loss_squared, 5.0, y_sample):.3f}")

In [ ]:
# Минимизируем Q(c) по константе c для каждой потери
grid = np.linspace(y_sample.min(), y_sample.max(), 600)
optimum = {}
for loss, name in [(loss_squared, "квадратичная"), (loss_absolute, "абсолютная"),
                   (loss_quantile, "квантильная, q=0.75")]:
    risks = np.array([empirical_risk(loss, c, y_sample) for c in grid])
    optimum[name] = grid[risks.argmin()]

for name, c in optimum.items():
    print(f"{name:22s}: c* = {c:6.3f}")
print(f"\nсреднее выборки  = {y_sample.mean():6.3f}")
print(f"медиана          = {np.median(y_sample):6.3f}")
print(f"квантиль 0.75    = {np.quantile(y_sample, 0.75):6.3f}")

> **Вывод.** Какой выборочной характеристике равен $c^*$ для каждой потери? Продифференцируйте $Q(c)$ по $c$ для квадратичной потери и убедитесь.
>
> *(ваш ответ здесь)*

---
# Часть 2. Проверяем утверждение 1.12

Утверждение 1.12: если $X^\ell$ — простая выборка, то
$\mathbb{E}\bigl[Q(a, X^\ell)\bigr] = R(a)$.

Ключевое слово — **алгоритм $a$ фиксирован до того, как увидена выборка**.
В доказательстве используется, что потери $\mathcal L(a, x_i, y_i)$ одинаково
распределены; если $a$ сам подобран по $X^\ell$, это рассуждение рушится.

Проверим оба случая на модели, где всё считается точно:
$x\sim U[0,1]$, $y = \theta_0 + \theta_1 x + \varepsilon$,
$\varepsilon\sim\mathcal N(0,\sigma^2)$. Для истинного алгоритма
$a^*(x) = \theta_0 + \theta_1 x$ истинный риск равен в точности
$R(a^*) = \mathbb{E}\varepsilon^2 = \sigma^2$.

In [ ]:
THETA_TRUE = np.array([2.0, 3.0])     # свободный член и наклон
SIGMA, ELL, N_REPEAT = 0.8, 30, 3000


def sample(n, generator):
    """Простая выборка (опр. 1.7) из распределения p(x, y)."""
    x = generator.uniform(0, 1, n)
    X = np.column_stack([np.ones(n), x])
    return X, X @ THETA_TRUE + generator.normal(0, SIGMA, n)


print(f"истинный риск R(a*) = sigma^2 = {SIGMA ** 2:.4f}")

### Задание 2.1. Фиксированный алгоритм

Повторим эксперимент $N$ раз: генерируем выборку и считаем $Q(a^*, X^\ell)$
для **истинного** алгоритма — того самого, что был известен до всех выборок.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

gen = np.random.default_rng(RANDOM_STATE)
Q_fixed = np.empty(N_REPEAT)
for t in range(N_REPEAT):
    X, y = sample(ELL, gen)
    # TODO (1 строка): Q для ИСТИННОГО алгоритма, то есть предсказания X @ THETA_TRUE
    Q_fixed[t] = ...

print(f"R(a*)             = {SIGMA ** 2:.4f}")
print(f"среднее Q         = {Q_fixed.mean():.4f}")
print(f"смещение          = {Q_fixed.mean() - SIGMA ** 2:+.4f}")

In [ ]:
# Теперь алгоритм подбирается ПО ТОЙ ЖЕ выборке (принцип ERM)
gen = np.random.default_rng(RANDOM_STATE + 1)
X_big, y_big = sample(200_000, gen)          # «генеральная совокупность» для оценки R

Q_train, R_hat = np.empty(N_REPEAT), np.empty(N_REPEAT)
for t in range(N_REPEAT):
    X, y = sample(ELL, gen)
    theta = np.linalg.lstsq(X, y, rcond=None)[0]
    Q_train[t] = empirical_risk(loss_squared, X @ theta, y)
    R_hat[t] = empirical_risk(loss_squared, X_big @ theta, y_big)

In [ ]:
p = 2                                        # число параметров модели
print(f"sigma^2                    = {SIGMA ** 2:.4f}")
print(f"среднее Q(a_hat, X^l)      = {Q_train.mean():.4f}"
      f"   теория sigma^2(1 - p/l) = {SIGMA ** 2 * (1 - p / ELL):.4f}")
print(f"среднее R(a_hat)           = {R_hat.mean():.4f}"
      f"   теория sigma^2(1 + p/l) = {SIGMA ** 2 * (1 + p / ELL):.4f}")
print(f"зазор R - Q                = {(R_hat - Q_train).mean():.4f}"
      f"   теория 2 sigma^2 p/l    = {2 * SIGMA ** 2 * p / ELL:.4f}")

In [ ]:
fig, ax = plt.subplots()
bins = np.linspace(0, np.percentile(R_hat, 99.5), 60)
ax.hist(Q_train, bins=bins, alpha=0.65, density=True, label=r"$Q$ на обучающей")
ax.hist(R_hat, bins=bins, alpha=0.65, density=True, label=r"$R$ истинный")
ax.axvline(SIGMA ** 2, color="black", lw=2, label=r"$\sigma^2$")
ax.set_xlabel("риск"); ax.set_ylabel("плотность"); ax.legend()
ax.set_title("Алгоритм выбран по выборке: обучающая ошибка смещена вниз")
plt.tight_layout(); plt.show()

> **Вывод.** Для фиксированного алгоритма смещения нет, для обученного — есть. В каком месте доказательства утверждения 1.12 ломается рассуждение? Как зазор зависит от $\ell$ и числа параметров $p$?
>
> *(ваш ответ здесь)*

---
# Часть 3. МНК: воспроизводим пример из конспекта

Теорема 1.20: если $X^{\mathsf T}X$ невырождена, то
$\theta^* = (X^{\mathsf T}X)^{-1}X^{\mathsf T}y$.

Формулу с обратной матрицей **не реализуют буквально** — решают систему.
В примере 1.21 конспекта МНК применён к выборке из восьми кошек
($f_1$ — банок корма в день, $f_2$ — возраст, $y$ — вес) и получено
$\theta^*\approx(2.201,\ 0.893,\ 0.114)$. Проверим.

> **Напоминание — число обусловленности.** $\mathrm{cond}(A)$ — отношение наибольшего сингулярного числа $A$ к
> наименьшему. Практический смысл: если $\mathrm{cond}(A)=10^{k}$, при решении
> системы с матрицей $A$ теряется около $k$ верных десятичных знаков, а
> $\mathrm{cond}=\infty$ означает вырожденность. Ключевой факт этой части:
> $\mathrm{cond}(X^{\mathsf T}X) = \mathrm{cond}(X)^2$ — переходя к нормальным
> уравнениям, мы **удваиваем** потерю точности. В NumPy: `np.linalg.cond(A)`.

In [ ]:
f1 = np.array([1, 2, 2, 3, 2, 3, 1, 4], dtype=float)
f2 = np.array([1, 2, 3, 5, 7, 10, 2, 8], dtype=float)
y_cats = np.array([3.0, 4.2, 4.5, 5.5, 4.8, 6.0, 3.4, 6.6])
X_cats = np.column_stack([np.ones(8), f1, f2])      # столбец единиц под theta_0

theta_solve = np.linalg.solve(X_cats.T @ X_cats, X_cats.T @ y_cats)
theta_lstsq = np.linalg.lstsq(X_cats, y_cats, rcond=None)[0]

print(f"нормальные уравнения (solve): {np.round(theta_solve, 4)}")
print(f"SVD-решение (lstsq)         : {np.round(theta_lstsq, 4)}")
print(f"в конспекте                 : [2.201  0.893  0.114]")

In [ ]:
# Почему не считают (X^T X)^{-1} явно: переход к нормальным уравнениям
# возводит число обусловленности в квадрат.
print(f"cond(X)      = {np.linalg.cond(X_cats):10.1f}")
print(f"cond(X^T X)  = {np.linalg.cond(X_cats.T @ X_cats):10.1f}"
      f"   = cond(X)^2 = {np.linalg.cond(X_cats) ** 2:.1f}")
print(f"\nQ(theta*) = {empirical_risk(loss_squared, X_cats @ theta_solve, y_cats):.6f}")

> **Вывод.** Совпало ли с конспектом? Что означает $\mathrm{cond}(X^{\mathsf T}X) = \mathrm{cond}(X)^2$ для точности вычислений?
>
> *(ваш ответ здесь)*

---
# Часть 4. Полиномы: первое переобучение

Как отмечено в примере 1.4 конспекта, признаками линейной модели могут служить
любые функции исходных данных: при $f_j(x) = x^{\,j-1}$ та же формула задаёт
полином. Модель остаётся **линейной по параметрам**, весь аппарат МНК применим.

Посмотрим, что происходит с ростом степени.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

true_f = lambda t: np.sin(3 * t) + 0.5 * t
x_tr = np.sort(rng.uniform(-1, 1, 30))
x_te = np.sort(rng.uniform(-1, 1, 500))
y_tr = true_f(x_tr) + rng.normal(0, 0.25, 30)
y_te = true_f(x_te) + rng.normal(0, 0.25, 500)

fit_poly = lambda d: make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(
    x_tr[:, None], y_tr)

### Задание 4.1. Ошибка на обучении и на контроле

Для каждой степени посчитайте эмпирический риск на обучающей выборке и на
контрольной. Обратите внимание: контрольная выборка в обучении не участвовала.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

degrees = [1, 3, 5, 7, 9, 12, 15]
rows = []
for d in degrees:
    model = fit_poly(d)
    # TODO (2 строки): Q на обучающей (x_tr, y_tr) и на контрольной (x_te, y_te)
    rows.append({"степень": d, "Q обучающая": ..., "Q контрольная": ...})
table = pd.DataFrame(rows).set_index("степень")
display(table.round(4))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
grid = np.linspace(-1, 1, 400)
ax1.scatter(x_tr, y_tr, s=25, color="black", zorder=3, label="обучающая выборка")
ax1.plot(grid, true_f(grid), "k--", lw=1.5, label="истинная зависимость")
for d in (1, 5, 15):
    ax1.plot(grid, fit_poly(d).predict(grid[:, None]), lw=1.8, label=f"степень {d}")
ax1.set_ylim(y_tr.min() - 1, y_tr.max() + 1); ax1.legend(fontsize=8)
ax1.set_xlabel("$x$"); ax1.set_ylabel("$y$"); ax1.set_title("Полиномы разных степеней")

ax2.plot(table.index, table["Q обучающая"], "o-", lw=2, label="обучающая")
ax2.plot(table.index, table["Q контрольная"], "s-", lw=2, label="контрольная")
ax2.set_yscale("log"); ax2.set_xlabel("степень"); ax2.set_ylabel("$Q$")
ax2.set_title("Риск против сложности модели"); ax2.legend()
plt.tight_layout(); plt.show()

> **Вывод.** Как ведут себя обе ошибки с ростом степени? Какая степень оптимальна и можно ли было выбрать её по обучающей ошибке?
>
> *(ваш ответ здесь)*

---
# Часть 5. Своя выборка

Применим МНК к индивидуальным данным. Функция `load_personal` повторяет
предобработку занятия 1, чтобы результат не зависел от аккуратности вашего кода.

In [ ]:
data = load_personal(variant)
Xtr, Xte = data["X_train"], data["X_test"]
ytr, yte = data["y_train"], data["y_test"]
print(f"{data['domain']}: {data['task']}, обучающая {Xtr.shape}, контрольная {Xte.shape}")

model = LinearRegression().fit(Xtr, ytr)
const = ytr.mean()

q_const = empirical_risk(loss_squared, const, yte)
q_model = empirical_risk(loss_squared, model.predict(Xte), yte)
print(f"\nQ на контроле: константа {q_const:.4g} | МНК {q_model:.4g}")
print(f"R^2 = 1 - Q_МНК / Q_конст = {1 - q_model / q_const:.4f}")

> **Вывод.** Насколько МНК лучше константного прогноза? Что означает $R^2 = 0$?
>
> *(ваш ответ здесь)*

## Итоги занятия

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Утверждение 1.12 говорит, что $\mathbb{E}[Q] = R$. Почему это не противоречит переобучению? Сформулируйте условие, при котором равенство верно.
2. Вы получили $Q(\hat a, X^\ell) = 0$. Что это говорит о качестве на новых объектах? Приведите алгоритм с нулевым эмпирическим риском и максимально плохим истинным.
3. Почему $\theta^* = (X^{\mathsf T}X)^{-1}X^{\mathsf T}y$ — правильная формула, но неправильная реализация?
4. Модель полиномов степени $\le d$ вложена в модель степени $\le d+1$. Докажите, что минимум эмпирического риска по большей модели не больше. Почему из этого **не** следует, что большая модель лучше?

---

**Дома:** откройте `lab02_homework.ipynb` — там три задачи: матричное дифференцирование, пять способов решить МНК и связь функции потерь с распределением шума.